In [1]:
# remember to resteart kernell
# ! pip install -e ../../wazeasy
# ! pip install pandas "dask[complete]" pyarrow plotly
# ! pip install --upgrade s3fs
# ! pip install geopandas
# ! pip install seaborn
# ! pip install folium matplotlib mapclassifys
# ! pip install h3
# ! pip install dask-geopandas
# ! pip install folium matplotlib mapclassify
!rm -rf /tmp/*

In [2]:
# import os
# os.environ['AWS_CA_BUNDLE'] = '/Users/mtadeo/Documents/gitrepo/cacert_new.pem'
# os.environ['REQUESTS_CA_BUNDLE'] = '/Users/mtadeo/Documents/gitrepo/cacert_new.pem'

In [3]:
import pandas as pd
import dask.dataframe as dd
import dask_geopandas
from dask import delayed, compute
from wazeasy import utils, plots, reports
import geopandas as gpd
from shapely import Polygon
from h3 import LatLngPoly
import yaml
import altair as alt
alt.renderers.enable('mimetype')
import re

In [4]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [5]:
storage_options = {'profile': 'ddp',
    # 'client_kwargs': {
    #     'verify': '/Users/mtadeo/Documents/gitrepo/cacert_new.pem'
    #     }
                  }

# file_list = [
#     's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000000.parquet',
#     's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000001.parquet',
#     's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000002.parquet',
#     's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000003.parquet',
#     's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000004.parquet',
#     's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000005.parquet',
#     's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000006.parquet',
#     's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000007.parquet',
#     's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000008.parquet',
#     's3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.000000000009.parquet'
# ]
# ddf = dd.read_parquet(file_list, storage_options = storage_options, engine = 'pyarrow')
# ddf = dd.read_parquet('s3://wbg-waze/bq/IQ/682baghdad/jams/*.parquet', 
#                       storage_options = storage_options, engine = 'pyarrow')

# path = 's3://wbg-waze/bq/IQ/682baghdad/jams/'
# df = pd.read_parquet(path, storage_options=storage_options)

In [6]:
%%time
ddf = utils.load_data('s3://wbg-waze/bq/IQ/682baghdad/jams/*.parquet', 
                      storage_options = storage_options, file_type = 'parquet', 
                      filter_level_5 = True, 
                      usecols = ['ts', 'geoWKT', 'uuid', 'level','length'])
ddf = ddf.repartition(npartitions=80)
# ddf = ddf.repartition(npartitions=400)
# ddf = ddf.persist()
# ddf = ddf.compute()


CPU times: user 1.03 s, sys: 164 ms, total: 1.19 s
Wall time: 2.48 s


In [7]:
timezone = config['Baghdad']['timezone']['timezone_name']
geographies = config['Baghdad']['geographies']
projected_crs = config['Baghdad']['crs']['projected_crs']

In [8]:
dict_geogs = {geog: gpd.read_file(geog_data['path']) for geog, geog_data in geographies.items()}

In [9]:
ddf = utils.assign_geography_to_jams(ddf, projected_crs)

In [10]:
utils.handle_time(ddf, timezone)

In [11]:
def create_gdf(ddf):
    '''
    Create a Dask-Geopandas GeoDataFrame from a Dask DataFrame.

    Parameters:
    - ddf (DataFrame): The Dask DataFrame containing geographical data.

    Returns:
    - GeoDataFrame: A GeoDataFrame with the geometry column set.
    '''
    ddf['geometry'] = dask_geopandas.from_wkt(ddf['geoWKT'], crs='epsg:4326')
    gddf = dask_geopandas.from_dask_dataframe(ddf, geometry='geometry')
    gddf = gddf.set_crs("EPSG:4326")
    return gddf
    
def obtain_unique_jams_linestrings(ddf):
    '''
    Get unique jam linestrings to avoid overlaying the same linestring multiple times.

    Parameters:
    - ddf (DataFrame): The Dask DataFrame containing traffic jam data.

    Returns:
    - GeoDataFrame: A GeoDataFrame with unique jam linestrings.
    '''
    unique_geo = ddf[["geoWKT"]].drop_duplicates().reset_index(drop=True).reset_index()
    unique_geo = create_gdf(unique_geo)
    return unique_geo

def overlay_group(group, hexagons):
    '''
    Perform an overlay between layers for delayed processes.

    Parameters:
    - group (GeoDataFrame): A GeoDataFrame group to overlay.
    - hexagons (GeoDataFrame): A GeoDataFrame of hexagons to overlay with.

    Returns:
    - GeoDataFrame: The result of the overlay operation.
    '''
    result = gpd.overlay(group, hexagons, how = 'intersection')
    return result

def parallelized_overlay(ddf, aggregation_geog):
    '''
    Parallelize overlay by groups over some geometry.

    Parameters:
    - ddf (DataFrame): The Dask DataFrame containing traffic jam data.
    - aggregation_geog (GeoDataFrame): The geographical areas for aggregation.

    Returns:
    - GeoDataFrame: The result of the parallelized overlay operation.
    '''
    unique_geo = obtain_unique_jams_linestrings(ddf).persist()
    delayed_process_group = delayed(overlay_group)
    groups = [unique_geo.get_partition(i) for i in range(unique_geo.npartitions)]
    tasks = [delayed_process_group(group, aggregation_geog) for group in groups]
    results = compute(*tasks)
    final_result = gpd.GeoDataFrame(pd.concat(results, ignore_index=True))
    return final_result

def distribute_jams_over_aggregation_geom(gddf, ddf, projected_crs):
    '''
    Distribute jams over aggregation geometry.

    Parameters:
    - gddf (GeoDataFrame): The GeoDataFrame with jams and geometry.
    - ddf (DataFrame): The Dask DataFrame containing traffic jam data.
    - projected_crs (str): The coordinate reference system for projection.

    Returns:
    - DataFrame: A DataFrame with jams distributed over the aggregation geometry.
    '''
    gddf = gddf.to_crs(projected_crs)
    gddf['length_in_geom'] = gddf['geometry'].length
    df = dd.from_pandas(gddf)
    merge = ddf.merge(df, left_on = 'geoWKT', right_on = 'geoWKT', how = 'left')   
    return merge

def classify_jam_by_region(ddf, geogs, year, month, projected_crs, dow = None):
    '''It is important to filter the dataset as much as it can be filtered before the spatial operation'''
    # start_date = dt(year, month, 1)
    # if month == 12:
    #     end_date = dt(year + 1, 1, 1) - timedelta(days=1)
    # else:
    #     end_date = dt(year, month + 1, 1) - timedelta(days=1)
    # date_range = [x.date() for x in pd.date_range(start_date, end_date)]
    # dates_of_interest = filter_date_range_by_dow(date_range, dow)
    
    # ddf_filtered = ddf[ddf['date'].isin(dates_of_interest)].copy()
    unique_jams_over_agg_geom = parallelized_overlay(ddf_filtered, geogs)
    jams_over_agg_geom = distribute_jams_over_aggregation_geom(unique_jams_over_agg_geom, ddf_filtered, projected_crs)    
    return jams_over_agg_geom

In [12]:
%%time
unique_geo = obtain_unique_jams_linestrings(ddf).persist()

CPU times: user 8min 55s, sys: 5min 14s, total: 14min 10s
Wall time: 3min 58s


In [13]:
aggregation_geog = dict_geogs['admin1'][['Region', 'geometry']]

In [14]:
%%time
delayed_process_group = delayed(overlay_group)
groups = [unique_geo.get_partition(i) for i in range(unique_geo.npartitions)]
tasks = [delayed_process_group(group, aggregation_geog) for group in groups]
results = compute(*tasks)
final_result = gpd.GeoDataFrame(pd.concat(results, ignore_index=True))

CPU times: user 3min 6s, sys: 63.7 ms, total: 3min 6s
Wall time: 6.84 s


In [15]:
%%time 
gddf = final_result.to_crs(projected_crs)
gddf['length_in_geom'] = gddf['geometry'].length

CPU times: user 17.9 s, sys: 90.5 ms, total: 18 s
Wall time: 18 s


In [16]:
%%time
df = dd.from_pandas(gddf)
merge = ddf.merge(df, left_on = 'geoWKT', right_on = 'geoWKT', how = 'left') 

CPU times: user 1.13 s, sys: 68.6 ms, total: 1.2 s
Wall time: 1.2 s


In [17]:
aggregation_geog.head()

,Region,geometry
0,Dulaim|Ramadi,"MULTIPOLYGON (((43.45248 32.58043, 43.44468 32..."
1,Basra|Bassora,"MULTIPOLYGON (((47.88775 30.03484, 47.83886 30..."
2,Al-Muthanna,"MULTIPOLYGON (((46.54779 30.54214, 46.56071 30..."
3,Diwaniyah,"MULTIPOLYGON (((44.92429 31.25666, 44.89528 31..."
4,NA,"MULTIPOLYGON (((42.8209 30.64079, 42.84176 30...."


In [19]:
%%time
merge.groupby('Region')['length_in_geom'].sum().compute()

Region
Dulaim|Ramadi              4.259815e+09
Salah ad-Din|Salaheddin    1.965831e+09
Babylon|Hilla              3.805110e+09
NA                         7.877133e+09
Diwaniyah                  1.032734e+06
Kut|Kut-al-Imara           4.113483e+08
Bagd|Bagdad                1.937169e+11
Name: length_in_geom, dtype: float64

In [11]:
unique_geo = ddf[["geoWKT"]].drop_duplicates().reset_index(drop=True).reset_index()

In [22]:
import dask_geopandas
unique_geo = create_gdf(unique_geo)

In [23]:
%%time
unique_geo = unique_geo.persist()

In [ ]:
def sjoin_group(group, hexagons):
    '''
    Perform an overlay between layers for delayed processes.

    Parameters:
    - group (GeoDataFrame): A GeoDataFrame group to overlay.
    - hexagons (GeoDataFrame): A GeoDataFrame of hexagons to overlay with.

    Returns:
    - GeoDataFrame: The result of the overlay operation.
    '''
    result = gpd.overlay(group, hexagons, how = 'intersection')
    return result

In [19]:
test.compute().memory_usage(deep=True).sum() / 1e9

1.11593628

In [12]:
# reports.run_basic_report(ddf, '2023-01-01', '2024-12-31')

In [13]:
# ddf = utils.assign_geography_to_jams(ddf, projected_crs, dict_geogs)

In [14]:
# import shapely
# import geopandas as gpd
# import dask_geopandas
# from shapely import Point

# def extract_last_point(wkt_series):
#     geoms = shapely.from_wkt(wkt_series.values)       # vectorized, NumPy-backed
#     points = shapely.get_point(geoms, -1)              # vectorized, no Python loop
#     return gpd.GeoSeries(points, crs='epsg:4326', index=wkt_series.index)
# meta = gpd.GeoSeries([], dtype='geometry', name='geometry', crs='epsg:4326')
# ddf['geometry'] = ddf['geoWKT'].map_partitions(extract_last_point, meta=meta)
# test = ddf.persist()

# def extract_last_point_str(wkt_series):
#     # Pulls the last coordinate pair from a LINESTRING WKT
#     def last_coord(wkt):
#         coords = re.findall(r'[\d\.\-]+ [\d\.\-]+', wkt)
#         x, y = coords[-1].split()
#         return Point(float(x), float(y))
#     points = wkt_series.map(last_coord)
#     return gpd.GeoSeries(points, crs='epsg:4326', index=wkt_series.index)

# meta = gpd.GeoSeries([], dtype='geometry', name='geometry', crs='epsg:4326')
# ddf['geometry'] = ddf['geoWKT'].map_partitions(extract_last_point_str, meta=meta)


In [ ]:
%%time
test = ddf.persist()
# 16.45

In [ ]:
test.head()

In [14]:
gddf = dask_geopandas.from_dask_dataframe(ddf, geometry='geometry')

In [15]:
gddf = gddf.set_crs('epsg:4326')

In [16]:
gdf_area = dict_geogs['admin1'][['Region', 'geometry']]
_ = gdf_area.sindex
join = gddf.sjoin(gdf_area, how='inner', predicate='intersects')
gddf['Region'] = join['Region']
gddf = gddf.rename(columns = {'Region': 'Admin1'})

In [ ]:
gddf_persist = gddf.persist()

In [ ]:
15.27

In [ ]:
reports.run_geog_report(ddf, geographies, start_date = None, end_date = None)

# Accidents